In [7]:
import numpy as np
import math

**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [8]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box)
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`:

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function.
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule.

        gradInput = module.backward(input, gradOutput)
    """
    def __init__ (self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self,input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput


    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it.
        """

        # The easiest case:

        # self.output = input
        # return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input.
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # The easiest case:

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Module"

# Sequential container

**Define** a forward and backward pass procedures.

In [9]:
class Sequential(Module):
    """
         This class implements a container, which processes `input` data sequentially.

         `input` is processed by each module (layer) in self.modules consecutively.
         The resulting array is called `output`.
    """

    def __init__ (self):
        super(Sequential, self).__init__()
        self.modules = []

    def add(self, module):
        """
        Adds a module to the container.
        """
        self.modules.append(module)

    def updateOutput(self, input):
        """
        Basic workflow of FORWARD PASS:

            y_0    = module[0].forward(input)
            y_1    = module[1].forward(y_0)
            ...
            output = module[n-1].forward(y_{n-2})


        Just write a little loop.
        """

        output = input

        for module in self.modules:
            output = module.forward(output)

        self.output = output
        return self.output

    def backward(self, input, gradOutput):
        """
        Workflow of BACKWARD PASS:

            g_{n-1} = module[n-1].backward(y_{n-2}, gradOutput)
            g_{n-2} = module[n-2].backward(y_{n-3}, g_{n-1})
            ...
            g_1 = module[1].backward(y_0, g_2)
            gradInput = module[0].backward(input, g_1)


        !!!

        To ech module you need to provide the input, module saw while forward pass,
        it is used while computing gradients.
        Make sure that the input for `i-th` layer the output of `module[i]` (just the same input as in forward pass)
        and NOT `input` to this Sequential module.

        !!!

        """
        grad = gradOutput

        for i in range(len(self.modules) - 1, -1, -1):
            if i == 0:
                current_input = input
            else:
                current_input = self.modules[i - 1].output

            grad = self.modules[i].backward(current_input, grad)

        self.gradInput = grad
        return self.gradInput


    def zeroGradParameters(self):
        for module in self.modules:
            module.zeroGradParameters()

    def getParameters(self):
        """
        Should gather all parameters in a list.
        """
        return [x.getParameters() for x in self.modules]

    def getGradParameters(self):
        """
        Should gather all gradients w.r.t parameters in a list.
        """
        return [x.getGradParameters() for x in self.modules]

    def __repr__(self):
        string = "".join([str(x) + '\n' for x in self.modules])
        return string

    def __getitem__(self,x):
        return self.modules.__getitem__(x)

    def train(self):
        """
        Propagates training parameter through all modules
        """
        self.training = True
        for module in self.modules:
            module.train()

    def evaluate(self):
        """
        Propagates training parameter through all modules
        """
        self.training = False
        for module in self.modules:
            module.evaluate()

# Layers

## 1 (0.2). Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [10]:
class Linear(Module):
    """
    A module which applies a linear transformation
    A common name is fully-connected layer, InnerProductLayer in caffe.

    The module should work with 2D input of shape (n_samples, n_feature).
    """
    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        # This is a nice initialization
        stdv = 1./np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size = (n_out, n_in))
        self.b = np.random.uniform(-stdv, stdv, size = n_out)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        self.output = input @ self.W.T + self.b
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput @ self.W
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradW += gradOutput.T @ input
        self.gradb += np.sum(gradOutput, axis=0)

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' %(s[1],s[0])
        return q

## 2. (0.2) SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [11]:
class SoftMax(Module):
    def __init__(self):
        super(SoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))

        exp_input = np.exp(self.output)
        self.output = exp_input / np.sum(exp_input, axis=1, keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        s = self.output
        dot = np.sum(gradOutput * s, axis=1, keepdims=True)
        self.gradInput = s * (gradOutput - dot)
        return self.gradInput

    def __repr__(self):
        return "SoftMax"

## 3. (0.2) LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [12]:
class LogSoftMax(Module):
    def __init__(self):
        super(LogSoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))

        log_sum_exp = np.log(np.sum(np.exp(self.output), axis=1, keepdims=True))
        self.output = self.output - log_sum_exp
        return self.output

    def updateGradInput(self, input, gradOutput):
        softmax = np.exp(self.output)
        self.gradInput = gradOutput - softmax * np.sum(gradOutput, axis=1, keepdims=True)
        return self.gradInput

    def __repr__(self):
        return "LogSoftMax"

## 4. (0.3) Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance:
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance.

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [13]:
class BatchNormalization(Module):
    EPS = 1e-3
    def __init__(self, alpha = 0.):
        super(BatchNormalization, self).__init__()
        self.alpha = alpha
        self.moving_mean = None
        self.moving_variance = None

    def updateOutput(self, input):
        if self.moving_mean is None:
            self.moving_mean = np.zeros(input.shape[1], dtype=input.dtype)

        if self.moving_variance is None:
            self.moving_variance = np.ones(input.shape[1], dtype=input.dtype)

        if self.training:
            self.batch_mean = np.mean(input, axis=0)
            self.batch_variance = np.var(input, axis=0)

            self.output = (input - self.batch_mean) / np.sqrt(self.batch_variance + self.EPS)

            self.moving_mean = self.moving_mean * self.alpha + self.batch_mean * (1 - self.alpha)
            self.moving_variance = self.moving_variance * self.alpha + self.batch_variance * (1 - self.alpha)
        else:
            self.output = (input - self.moving_mean) / np.sqrt(self.moving_variance + self.EPS)

        return self.output

    def updateGradInput(self, input, gradOutput):
        if self.training:
            batch_size = input.shape[0]
            normalized_input = (input - self.batch_mean) / np.sqrt(self.batch_variance + self.EPS)
            std_inv = 1.0 / np.sqrt(self.batch_variance + self.EPS)

            self.gradInput = (1.0 / batch_size) * std_inv * (
                batch_size * gradOutput
                - np.sum(gradOutput, axis=0)
                - normalized_input * np.sum(gradOutput * normalized_input, axis=0)
            )
        else:
            self.gradInput = gradOutput / np.sqrt(self.moving_variance + self.EPS)

        return self.gradInput

    def __repr__(self):
        return "BatchNormalization"

In [14]:
class ChannelwiseScaling(Module):
    """
       Implements linear transform of input y = \gamma * x + \beta
       where \gamma, \beta - learnable vectors of length x.shape[-1]
    """
    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        stdv = 1./np.sqrt(n_out)
        self.gamma = np.random.uniform(-stdv, stdv, size=n_out)
        self.beta = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma + self.beta
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradBeta = np.sum(gradOutput, axis=0)
        self.gradGamma = np.sum(gradOutput*input, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. (0.3) Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [15]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()

        self.p = p
        self.mask = None

    def updateOutput(self, input):
        if self.training:
            self.mask = (np.random.rand(*input.shape) >= self.p).astype(input.dtype) / (1.0 - self.p)
            self.output = input * self.mask
        else:
            self.output = input.copy()
        return self.output

    def updateGradInput(self, input, gradOutput):
        if self.training:
            self.gradInput = gradOutput * self.mask
        else:
            self.gradInput = gradOutput.copy()
        return self.gradInput

    def __repr__(self):
        return "Dropout"

#6. (2.0) Conv2d
Implement [**Conv2d**](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Use only this list of parameters: (in_channels, out_channels, kernel_size, stride, padding, bias, padding_mode) and fix dilation=1 and groups=1.

In [16]:
class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size,
        stride=1, padding=0, bias=True, padding_mode='zeros'):
        super(Conv2d, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.bias = bias
        self.padding_mode = padding_mode

    def updateOutput(self, input):
        def _pair(x):
            if isinstance(x, tuple):
                return x
            return (x, x)

        def _same_pad(size, kernel, stride):
            out = int(np.ceil(size / stride))
            total = max((out - 1) * stride + kernel - size, 0)
            before = total // 2
            after = total - before
            return before, after

        kH, kW = _pair(self.kernel_size)
        sH, sW = _pair(self.stride)

        N, C, H, W = input.shape
        out_channels, in_channels, _, _ = self.weight.shape

        if isinstance(self.padding, str):
            if self.padding != 'same':
                raise ValueError("Only padding='same' string is supported")
            pad_top, pad_bottom = _same_pad(H, kH, sH)
            pad_left, pad_right = _same_pad(W, kW, sW)
        else:
            pH, pW = _pair(self.padding)
            pad_top = pad_bottom = pH
            pad_left = pad_right = pW

        if self.padding_mode == 'zeros':
            padded_input = np.pad(
                input,
                ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
                mode='constant'
            )
            index_map = None
        elif self.padding_mode == 'reflect':
            padded_input = np.pad(
                input,
                ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
                mode='reflect'
            )
            base_idx = np.arange(H * W).reshape(H, W)
            index_map = np.pad(
                base_idx,
                ((pad_top, pad_bottom), (pad_left, pad_right)),
                mode='reflect'
            )
        elif self.padding_mode == 'replicate':
            padded_input = np.pad(
                input,
                ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
                mode='edge'
            )
            base_idx = np.arange(H * W).reshape(H, W)
            index_map = np.pad(
                base_idx,
                ((pad_top, pad_bottom), (pad_left, pad_right)),
                mode='edge'
            )
        else:
            raise ValueError("Unsupported padding_mode")

        H_pad, W_pad = padded_input.shape[2], padded_input.shape[3]
        out_H = (H_pad - kH) // sH + 1
        out_W = (W_pad - kW) // sW + 1

        self.output = np.zeros((N, out_channels, out_H, out_W), dtype=input.dtype)

        for n in range(N):
            for oc in range(out_channels):
                for oh in range(out_H):
                    h_start = oh * sH
                    h_end = h_start + kH
                    for ow in range(out_W):
                        w_start = ow * sW
                        w_end = w_start + kW

                        window = padded_input[n, :, h_start:h_end, w_start:w_end]
                        self.output[n, oc, oh, ow] = np.sum(window * self.weight[oc])

                if np.ndim(self.bias) == 1:
                    self.output[n, oc] += self.bias[oc]

        self._conv_padded_input = padded_input
        self._conv_index_map = index_map
        self._conv_padding = (pad_top, pad_bottom, pad_left, pad_right)

        return self.output

    def updateGradInput(self, input, gradOutput):
        def _pair(x):
            if isinstance(x, tuple):
                return x
            return (x, x)

        kH, kW = _pair(self.kernel_size)
        sH, sW = _pair(self.stride)

        N, C, H, W = input.shape
        out_channels, in_channels, _, _ = self.weight.shape

        padded_input = self._conv_padded_input
        index_map = self._conv_index_map
        pad_top, pad_bottom, pad_left, pad_right = self._conv_padding

        H_pad, W_pad = padded_input.shape[2], padded_input.shape[3]
        out_H, out_W = gradOutput.shape[2], gradOutput.shape[3]

        grad_padded = np.zeros_like(padded_input)

        for n in range(N):
            for oc in range(out_channels):
                for oh in range(out_H):
                    h_start = oh * sH
                    h_end = h_start + kH
                    for ow in range(out_W):
                        w_start = ow * sW
                        w_end = w_start + kW

                        grad_padded[n, :, h_start:h_end, w_start:w_end] += (
                            gradOutput[n, oc, oh, ow] * self.weight[oc]
                        )

        if self.padding_mode == 'zeros':
            self.gradInput = grad_padded[:, :, pad_top:pad_top + H, pad_left:pad_left + W]
        else:
            self.gradInput = np.zeros((N, C, H, W), dtype=input.dtype)

            flat_map = index_map.reshape(-1)
            grad_padded_flat = grad_padded.reshape(N, C, -1)

            for p, orig in enumerate(flat_map):
                r = orig // W
                c = orig % W
                self.gradInput[:, :, r, c] += grad_padded_flat[:, :, p]

        return self.gradInput

    def __repr__(self):
        return "Conv2d"

#7. (0.5) Implement [**MaxPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [**AvgPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html). Use only parameters like kernel_size, stride, padding (negative infinity for maxpool and zero for avgpool) and other parameters fixed as in framework.

In [17]:
class MaxPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(MaxPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        def _pair(x):
            if isinstance(x, tuple):
                return x
            return (x, x)

        kH, kW = _pair(self.kernel_size)
        sH, sW = _pair(self.stride)
        pH, pW = _pair(self.padding)

        N, C, H, W = input.shape

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=-np.inf
        )

        H_pad, W_pad = padded_input.shape[2], padded_input.shape[3]
        out_H = (H_pad - kH) // sH + 1
        out_W = (W_pad - kW) // sW + 1

        self.output = np.zeros((N, C, out_H, out_W), dtype=input.dtype)
        self.max_indices = np.zeros((N, C, out_H, out_W, 2), dtype=np.int64)

        for n in range(N):
            for c in range(C):
                for oh in range(out_H):
                    h_start = oh * sH
                    h_end = h_start + kH
                    for ow in range(out_W):
                        w_start = ow * sW
                        w_end = w_start + kW

                        window = padded_input[n, c, h_start:h_end, w_start:w_end]
                        self.output[n, c, oh, ow] = np.max(window)

                        max_pos = np.unravel_index(np.argmax(window), window.shape)
                        self.max_indices[n, c, oh, ow, 0] = h_start + max_pos[0]
                        self.max_indices[n, c, oh, ow, 1] = w_start + max_pos[1]

        self.padded_shape = padded_input.shape
        return self.output

    def updateGradInput(self, input, gradOutput):
        def _pair(x):
            if isinstance(x, tuple):
                return x
            return (x, x)

        pH, pW = _pair(self.padding)

        N, C, H, W = input.shape
        _, _, out_H, out_W = gradOutput.shape

        grad_padded = np.zeros(self.padded_shape, dtype=input.dtype)

        for n in range(N):
            for c in range(C):
                for oh in range(out_H):
                    for ow in range(out_W):
                        h_idx, w_idx = self.max_indices[n, c, oh, ow]
                        grad_padded[n, c, h_idx, w_idx] += gradOutput[n, c, oh, ow]

        self.gradInput = grad_padded[:, :, pH:pH + H, pW:pW + W]
        return self.gradInput

    def __repr__(self):
        return "MaxPool2d"

class AvgPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(AvgPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        def _pair(x):
            if isinstance(x, tuple):
                return x
            return (x, x)

        kH, kW = _pair(self.kernel_size)
        sH, sW = _pair(self.stride)
        pH, pW = _pair(self.padding)

        N, C, H, W = input.shape

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=0.0
        )

        H_pad, W_pad = padded_input.shape[2], padded_input.shape[3]
        out_H = (H_pad - kH) // sH + 1
        out_W = (W_pad - kW) // sW + 1

        self.output = np.zeros((N, C, out_H, out_W), dtype=input.dtype)

        for n in range(N):
            for c in range(C):
                for oh in range(out_H):
                    h_start = oh * sH
                    h_end = h_start + kH
                    for ow in range(out_W):
                        w_start = ow * sW
                        w_end = w_start + kW

                        window = padded_input[n, c, h_start:h_end, w_start:w_end]
                        self.output[n, c, oh, ow] = np.mean(window)

        self.padded_shape = padded_input.shape
        return self.output

    def updateGradInput(self, input, gradOutput):
        def _pair(x):
            if isinstance(x, tuple):
                return x
            return (x, x)

        kH, kW = _pair(self.kernel_size)
        sH, sW = _pair(self.stride)
        pH, pW = _pair(self.padding)

        N, C, H, W = input.shape
        _, _, out_H, out_W = gradOutput.shape

        grad_padded = np.zeros(self.padded_shape, dtype=input.dtype)
        scale = 1.0 / (kH * kW)

        for n in range(N):
            for c in range(C):
                for oh in range(out_H):
                    h_start = oh * sH
                    h_end = h_start + kH
                    for ow in range(out_W):
                        w_start = ow * sW
                        w_end = w_start + kW

                        grad_padded[n, c, h_start:h_end, w_start:w_end] += gradOutput[n, c, oh, ow] * scale

        self.gradInput = grad_padded[:, :, pH:pH + H, pW:pW + W]
        return self.gradInput

    def __repr__(self):
        return "AvgPool2d"

#8. (0.3) Implement **GlobalMaxPool2d** and **GlobalAvgPool2d**. They do not have testing and parameters are up to you but they must aggregate information within channels. Write test functions for these layers on your own.

In [18]:
class GlobalAvgPool2d(Module):
    def __init__(self):
        super(GlobalAvgPool2d, self).__init__()

    def updateOutput(self, input):
        self.output = np.mean(input, axis=(2, 3))
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = input.shape
        self.gradInput = np.ones_like(input) * gradOutput[:, :, None, None] / (H * W)
        return self.gradInput

    def __repr__(self):
        return "GlobalAvgPool2d"

class GlobalMaxPool2d(Module):
    def __init__(self):
        super(GlobalMaxPool2d, self).__init__()

    def updateOutput(self, input):
        N, C, H, W = input.shape

        input_flat = input.reshape(N, C, -1)
        self.max_indices = np.argmax(input_flat, axis=2)
        self.output = np.max(input_flat, axis=2)

        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = input.shape
        self.gradInput = np.zeros_like(input)

        for n in range(N):
            for c in range(C):
                idx = self.max_indices[n, c]
                h = idx // W
                w = idx % W
                self.gradInput[n, c, h, w] = gradOutput[n, c]

        return self.gradInput

    def __repr__(self):
        return "GlobalMaxPool2d"

#9. (0.2) Implement [**Flatten**](https://pytorch.org/docs/stable/generated/torch.flatten.html)

In [19]:
class Flatten(Module):
    def __init__(self, start_dim=0, end_dim=-1):
        super(Flatten, self).__init__()

        self.start_dim = start_dim
        self.end_dim = end_dim

    def updateOutput(self, input):
        x = np.asarray(input)

        start = self.start_dim if self.start_dim >= 0 else x.ndim + self.start_dim
        end = self.end_dim if self.end_dim >= 0 else x.ndim + self.end_dim

        new_shape = (
            list(x.shape[:start]) +
            [int(np.prod(x.shape[start:end + 1]))] +
            list(x.shape[end + 1:])
        )
        
        self.output = x.reshape(new_shape)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput.reshape(input.shape)
        return self.gradInput

    def __repr__(self):
        return "Flatten"

# Activation functions

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**):

In [20]:
class ReLU(Module):
    def __init__(self):
         super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput , input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 10. (0.1) Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope.

In [21]:
class LeakyReLU(Module):
    def __init__(self, slope = 0.03):
        super(LeakyReLU, self).__init__()

        self.slope = slope

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.slope * input)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input > 0, 1.0, self.slope)
        return self.gradInput

    def __repr__(self):
        return "LeakyReLU"

## 11. (0.1) ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [22]:
class ELU(Module):
    def __init__(self, alpha = 1.0):
        super(ELU, self).__init__()

        self.alpha = alpha

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.alpha * (np.exp(input) - 1))
        return  self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input > 0, 1.0, self.alpha * np.exp(input))
        return self.gradInput

    def __repr__(self):
        return "ELU"

## 12. (0.1) SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [23]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        self.output = np.log1p(np.exp(-np.abs(input))) + np.maximum(input, 0)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        sigmoid = 1.0 / (1.0 + np.exp(-input))
        self.gradInput = gradOutput * sigmoid
        return self.gradInput

    def __repr__(self):
        return "SoftPlus"

#13. (0.2) Gelu
Implement [**Gelu**](https://pytorch.org/docs/stable/generated/torch.nn.GELU.html) activations.

In [24]:
class Gelu(Module):
    def __init__(self):
        super(Gelu, self).__init__()

    def updateOutput(self, input):
        erf_vec = np.vectorize(math.erf)
        self.output = 0.5 * input * (1.0 + erf_vec(input / np.sqrt(2.0)))
        return  self.output

    def updateGradInput(self, input, gradOutput):
        erf_vec = np.vectorize(math.erf)
        erf_term = erf_vec(input / np.sqrt(2.0))
        pdf_term = np.exp(-0.5 * input**2) / np.sqrt(2.0 * np.pi)
        derivative = 0.5 * (1.0 + erf_term) + input * pdf_term
        self.gradInput = gradOutput * derivative
        return self.gradInput

    def __repr__(self):
        return "Gelu"

# Criterions

Criterions are used to score the models answers.

In [25]:
class Criterion(object):
    def __init__ (self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [26]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target,2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput  = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 14. (0.2) Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula,
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [27]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15
    def __init__(self):
        a = super(ClassNLLCriterionUnstable, self)
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        self.output = -np.sum(target * np.log(input_clamp)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        self.gradInput = -(target / input_clamp) / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"

## 15. (0.3) Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [28]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        a = super(ClassNLLCriterion, self)
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = -np.sum(target * input) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput = -target / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterion"

1-я часть задания: реализация слоев, лосей и функций активации - 5 баллов. \\
2-я часть задания: реализация моделей на своих классах. Что должно быть:
  1. Выберите оптимизатор и реализуйте его, чтоб он работал с вами классами. - 1 балл.
  2. Модель для задачи мультирегрессии на выбраных вами данных. Использовать FCNN, dropout, batchnorm, MSE. Пробуйте различные фукнции активации. Для первой модели попробуйте большую, среднюю и маленькую модель. - 1 балл.
  3. Модель для задачи мультиклассификации на MNIST. Использовать свёртки, макспулы, флэттэны, софтмаксы - 1 балла.
  4. Автоэнкодер для выбранных вами данных. Должен быть на свёртках и полносвязных слоях, дропаутах, батчнормах и тд. - 2 балла. \\

Дополнительно в оценке каждой модели будет учитываться:
1. Наличие правильно выбранной метрики и лосс функции.
2. Отрисовка графиков лосей и метрик на трейне-валидации. Проверка качества модели на тесте.
3. Наличие шедулера для lr.
4. Наличие вормапа.
5. Наличие механизма ранней остановки и сохранение лучшей модели.
6. Свитч лося (метрики) и оптимайзера.

## 1

In [29]:
class Adam:
    def __init__(self, model, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.model = model
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        params = self.model.getParameters()
        self.m = self._zeros_like_structure(params)
        self.v = self._zeros_like_structure(params)

    def _zeros_like_structure(self, obj):
        if isinstance(obj, list):
            return [self._zeros_like_structure(x) for x in obj]
        else:
            return np.zeros_like(obj)

    def zero_grad(self):
        self.model.zeroGradParameters()

    def step(self):
        self.t += 1

        params = self.model.getParameters()
        grads = self.model.getGradParameters()

        self._step_recursive(params, grads, self.m, self.v)

    def _step_recursive(self, params, grads, m, v):
        if isinstance(params, list):
            for p, g, m_i, v_i in zip(params, grads, m, v):
                self._step_recursive(p, g, m_i, v_i)
        else:
            if grads is None:
                return
                
            m[...] = self.beta1 * m + (1.0 - self.beta1) * grads
            v[...] = self.beta2 * v + (1.0 - self.beta2) * (grads ** 2)

            m_hat = m / (1.0 - self.beta1 ** self.t)
            v_hat = v / (1.0 - self.beta2 ** self.t)

            params[...] = params - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)